In [0]:
# =============================================================================
# EA Real-Time Archive Backfill
# =============================================================================
# Notebook:   08_ea_rt_archive_backfill.py
# Schema:     prd_dash_lab.flood_forecasting_unrestricted
# Table:      ea_rt_readings_bronze
# Source:     EA Real-Time Flood Monitoring Archive
#             https://environment.data.gov.uk/flood-monitoring/archive
# Run:        ONE-TIME MANUAL RUN only. Not scheduled.
#
# Purpose:
#   Downloads the last 30 days of readings-full CSV files from the EA archive
#   and lands them as a partitioned Bronze Delta table. Each day's file
#   contains all readings for all stations and parameter types at 15-minute
#   resolution.
#
#   This notebook is the foundation for:
#     - 09_ea_rt_measure_register.py  (register built from distinct measures)
#     - 10_ea_rt_daily_archive.py     (daily incremental appends)
#
# Restart behaviour:
#   If the notebook fails mid-run, re-run it. Completed dates are detected
#   from existing partitions in the Bronze table itself -- no separate
#   checkpoint table is needed.
#
# Archive availability:
#   Each day's archive file is generated at 22:00 the following day to allow
#   late telemetry to arrive. The backfill therefore covers dates from
#   (today - 31) through (today - 2), giving 30 complete files.
#   Today and yesterday are excluded -- yesterday's file may not yet exist.
#
# CSV columns (readings-full format):
#   dateTime, date, measure, station, label, stationReference,
#   parameter, qualifier, datumType, period, unitName, valueType, value
#
# Attribution (OGL):
#   "this uses Environment Agency flood and river level data from the
#    real-time data API (Beta)"
# =============================================================================

# Standard library imports
import requests
import io
from datetime import datetime, timedelta, timezone, date

# Data handling
import pandas as pd

# PySpark
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType, DateType, IntegerType
)

# =============================================================================
# CONFIGURATION
# =============================================================================

CATALOG      = "prd_dash_lab"
SCHEMA       = "flood_forecasting_unrestricted"
BRONZE_TABLE = "ea_rt_readings_bronze"

FULL_BRONZE_NAME = f"{CATALOG}.{SCHEMA}.{BRONZE_TABLE}"

# EA archive base URL. Files follow the pattern readings-full-{YYYY-MM-DD}.csv
ARCHIVE_BASE = "https://environment.data.gov.uk/flood-monitoring/archive"

# How many days to backfill.
# Archive is available from (today - 31) through (today - 2).
BACKFILL_DAYS = 30

# Request timeout per file. Files are ~130MB -- allow generous time.
DOWNLOAD_TIMEOUT_SECONDS = 300


In [0]:
# =============================================================================
# STEP 1: CALCULATE THE DATE RANGE TO BACKFILL
# =============================================================================

today      = date.today()
end_date   = today - timedelta(days=2)   # yesterday may not yet be generated
start_date = today - timedelta(days=31)  # 30 full days

all_dates = [
    start_date + timedelta(days=i)
    for i in range((end_date - start_date).days + 1)
]

print(f"Backfill range: {start_date} to {end_date} ({len(all_dates)} dates)")


In [0]:
# =============================================================================
# STEP 2: DETECT ALREADY-COMPLETED DATES FROM BRONZE
# =============================================================================
# A date is complete if its partition already exists in the Bronze table.
# No separate checkpoint table is needed -- the Bronze table is the record.
# If Bronze does not exist yet, treat all dates as pending.

try:
    completed_dates = {
        row["reading_date"]
        for row in spark.sql(f"""
            SELECT DISTINCT reading_date
            FROM {FULL_BRONZE_NAME}
        """).collect()
    }
    print(f"Bronze table found. {len(completed_dates)} dates already present.")
except Exception:
    # Table does not exist yet -- first run
    completed_dates = set()
    print("Bronze table does not exist yet. All dates will be processed.")

dates_to_process = [d for d in all_dates if d not in completed_dates]
print(f"Dates remaining: {len(dates_to_process)}")

if not dates_to_process:
    print("All dates already present in Bronze. Nothing to do.")
    dbutils.notebook.exit("success")

In [0]:
# =============================================================================
# STEP 3: DEFINE THE BRONZE SCHEMA
# =============================================================================
# Matches the readings-full CSV columns exactly, renamed to snake_case.
# reading_date is the partition column.
# ingested_at records when this notebook wrote each row.

bronze_schema = StructType([
    StructField("date_time",         TimestampType(), nullable=True),
    StructField("reading_date",      DateType(),      nullable=False),
    StructField("measure_uri",       StringType(),    nullable=True),
    StructField("station_uri",       StringType(),    nullable=True),
    StructField("label",             StringType(),    nullable=True),
    StructField("station_reference", StringType(),    nullable=True),
    StructField("parameter",         StringType(),    nullable=True),
    StructField("qualifier",         StringType(),    nullable=True),
    StructField("datum_type",        StringType(),    nullable=True),
    StructField("period",            IntegerType(),   nullable=True),
    StructField("unit_name",         StringType(),    nullable=True),
    StructField("value_type",        StringType(),    nullable=True),
    StructField("value",             DoubleType(),    nullable=True),
    StructField("ingested_at",       TimestampType(), nullable=False),
])

# CSV column name -> Bronze column name
CSV_COLUMN_MAP = {
    "dateTime":         "date_time",
    "date":             "reading_date",
    "measure":          "measure_uri",
    "station":          "station_uri",
    "label":            "label",
    "stationReference": "station_reference",
    "parameter":        "parameter",
    "qualifier":        "qualifier",
    "datumType":        "datum_type",
    "period":           "period",
    "unitName":         "unit_name",
    "valueType":        "value_type",
    "value":            "value",
}


In [0]:
# =============================================================================
# STEP 4: DOWNLOAD AND LAND EACH DATE
# =============================================================================

now_utc = datetime.now(timezone.utc)

for i, target_date in enumerate(dates_to_process):

    date_str = target_date.strftime("%Y-%m-%d")
    url      = f"{ARCHIVE_BASE}/readings-full-{date_str}.csv"

    print(f"\n[{i+1}/{len(dates_to_process)}] {date_str}")
    print(f"  URL: {url}")

    # ------------------------------------------------------------------
    # Download
    # ------------------------------------------------------------------
    try:
        response = requests.get(url, timeout=DOWNLOAD_TIMEOUT_SECONDS)
        response.raise_for_status()
    except requests.exceptions.HTTPError:
        if response.status_code == 404:
            # No file for this date -- skip without failing the run
            print(f"  WARNING: No archive file for {date_str}. Skipping.")
            continue
        else:
            raise

    print(f"  Downloaded {len(response.content) / (1024*1024):.1f} MB")

    # ------------------------------------------------------------------
    # Parse CSV
    # ------------------------------------------------------------------
    # Read all columns as strings first; type casting happens in Spark
    pdf = pd.read_csv(
        io.BytesIO(response.content),
        dtype=str,
        low_memory=False
    )

    print(f"  Rows: {len(pdf):,}")

    # Rename to Bronze convention; drop any unexpected extra columns
    pdf = pdf.rename(columns=CSV_COLUMN_MAP)
    pdf = pdf[[c for c in CSV_COLUMN_MAP.values() if c in pdf.columns]]
    pdf["ingested_at"] = now_utc

    # ------------------------------------------------------------------
    # Convert to Spark and cast types
    # ------------------------------------------------------------------
    sdf = (
        spark.createDataFrame(pdf)
        .withColumn("date_time",    F.to_timestamp("date_time"))
        .withColumn("reading_date", F.to_date("reading_date"))
        .withColumn("period",       F.col("period").cast("integer"))
        .withColumn("value",        F.expr("try_cast(value AS DOUBLE)"))
        .withColumn("ingested_at",  F.col("ingested_at").cast("timestamp"))
    )

    # ------------------------------------------------------------------
    # Write to Bronze
    # replaceWhere overwrites this date's partition only, so a re-run
    # on a previously written date replaces rather than duplicates.
    # ------------------------------------------------------------------
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("replaceWhere", f"reading_date = '{date_str}'")
        .partitionBy("reading_date")
        .saveAsTable(FULL_BRONZE_NAME)
    )

    print(f"  Written to Bronze (partition: {date_str})")


In [0]:
# =============================================================================
# STEP 5: SUMMARY
# =============================================================================

total_rows = spark.sql(f"SELECT COUNT(*) AS n FROM {FULL_BRONZE_NAME}").collect()[0]["n"]

spark.sql(f"""
    SELECT
        MIN(reading_date)            AS earliest_date,
        MAX(reading_date)            AS latest_date,
        COUNT(DISTINCT reading_date) AS date_count,
        COUNT(DISTINCT station_reference) AS station_count,
        COUNT(DISTINCT measure_uri)  AS measure_count
    FROM {FULL_BRONZE_NAME}
""").show(truncate=False)

spark.sql(f"""
    SELECT parameter, COUNT(DISTINCT measure_uri) AS measures, COUNT(*) AS reading_count
    FROM {FULL_BRONZE_NAME}
    GROUP BY parameter
    ORDER BY reading_count DESC
""").show(truncate=False)

print(f"Total rows: {total_rows:,}")


In [0]:
# =============================================================================
# SIGNAL COMPLETION
# =============================================================================

print("Backfill complete.")
dbutils.notebook.exit("success")
